## Silver Layer

###### # -- full load done at very initial stage and disabling the full load for now, after this , incremantal load for for fact table and delta load for dimesion tables

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import *

### Full Load

In [0]:
## customer full load:

from pyspark.sql.functions import *

cust_df= spark.table("retailfashiondata.bronze.customer_data")

cust_df.write.mode("overwrite").saveAsTable("retailfashiondata.silver.customerTable")

##prod full load

prod_df= spark.table("retailfashiondata.bronze.product_data")

prod_df.write.mode("overwrite").saveAsTable("retailfashiondata.silver.productTable")



In [0]:
%sql
drop table retailfashiondata.silver.customerTable;
--drop table retailfashiondata.silver.productTable;
--drop table retailfashiondata.silver.salesTable;
--drop table retailfashiondata.silver.storeTable;

In [0]:
%sql
select * from retailfashiondata.silver.customerTable;
select * from retailfashiondata.silver.productTable;

In [0]:
%sql
-- Sales full data load
CREATE OR REPLACE TABLE retailfashiondata.silver.salesTable AS
SELECT * FROM retailfashiondata.bronze.sales_data;




In [0]:
%sql
select * from retailfashiondata.silver.salesTable;

In [0]:
%sql
CREATE OR REPLACE TABLE retailfashiondata.silver.storeTable AS
SELECT *
FROM retailfashiondata.bronze.store_data;

In [0]:
%sql
select * from retailfashiondata.silver.storeTable

### Incremental Load for Fact table (Sales_data)

In [0]:
last_modified_date = spark.sql("SELECT MAX(ingestion_time) FROM retailfashiondata.silver.salestable").collect()[0][0]

fact_sales = spark.table("retailfashiondata.bronze.sales_data")
fact_sales = fact_sales.filter(col("ingestion_time") > last_modified_date)

# Register as temp view so SQL cells can reference it
# sql only tries to find the unity catalog tables and temp views , for megre
fact_sales.createOrReplaceTempView("fact_sales")

display(last_modified_date)



In [0]:
%sql
Merge into retailfashiondata.silver.salestable as t
using fact_sales as s
on t.transaction_id = s.transaction_id
when Matched then Update set *
when Not matched then Insert *


## Delta Load for Dim tables 

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

df = spark.table("retailfashiondata.bronze.customer_data")

window_spec = Window.partitionBy("customer_id") \
                    .orderBy(col("ingestion_time").desc())

df_cust_dedup= (
    df.withColumn("rn", row_number().over(window_spec))
      .filter("rn = 1")
      .drop("rn")
)

df_cust_dedup.write.format("delta").mode("append").option("mergeSchema","true").saveAsTable("retailfashiondata.dedup_bronze_to_silver.cust_dim")


In [0]:
%sql
--merge customer table

MERGE WITH SCHEMA EVOLUTION
INTO retailfashiondata.silver.customertable t
USING retailfashiondata.dedup_bronze_to_silver.cust_dim as s
on t.customer_id = s.customer_id
when Matched then update set *
when Not Matched then insert *

In [0]:
%sql
select count(customer_id) from  retailfashiondata.silver.customertable ;

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

df = spark.table("retailfashiondata.bronze.product_data")

window_spec = Window.partitionBy("product_id") \
                    .orderBy(col("ingestion_time").desc())

df_prod_dedup= (
    df.withColumn("rn", row_number().over(window_spec))
      .filter("rn = 1")
      .drop("rn")
)

df_prod_dedup.format("delta").mode("append").option("mergeSchema","true").saveAsTable("retailfashiondata.dedup_bronze_to_silver.prod_dim")


In [0]:
%sql

MERGE WITH SCHEMA EVOLUTION
INTO retailfashiondata.silver.producttable t
USING retailfashiondata.dedup_bronze_to_silver.prod_dim as s
on t.product_id = s.product_id
when Matched then update set *
when Not Matched then insert *


In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

df = spark.table("retailfashiondata.bronze.store_data")

window_spec = Window.partitionBy("store_id") \
                    .orderBy(col("ingestion_time").desc())

df_store_dedup= (
    df.withColumn("rn", row_number().over(window_spec))
      .filter("rn = 1")
      .drop("rn")
)

df_store_dedup.format("delta").mode("append").option("mergeSchema","true").saveAsTable("retailfashiondata.dedup_bronze_to_silver.store_dim")

In [0]:
%sql
--merge customer table

Merge into retailfashiondata.silver.storetable as t
using retailfashiondata.dedup_bronze_to_silver.store_dim as s
on t.store_id = s.store_id
when Matched then update set *
when Not Matched then insert *

In [0]:
%sql
select * from retailfashiondata.silver.storetable;

##Transformation

In [0]:
%sql
 ---delete table retailfashiondata.silvertransformed.cust_dim;
-- delete table retailfashiondata.silvertransformed.prod_dim;
 --delete table retailfashiondata.silvertransformed.store_dim;
-- delete table retailfashiondata.silvertransformed.sales_dim

In [0]:
%sql
Create schema retailfashiondata.rescued_data

In [0]:
%sql
--delete from retailfashiondata.rescued_data.invalid_cust_dim;
--delete from retailfashiondata.SilverTransformed.cust_dim;

In [0]:
cust_dim= spark.table("retailfashiondata.silver.customertable")

## storing the _rescued_data to where it is not null(schema retailfashiondata.rescued_data)
invalid_cust_dim= cust_dim.filter(col("_rescued_data").isNotNull())

#write
(invalid_cust_dim.write
                .format("delta")
                .mode("append")
                .option("mergeSchema","true")
                .saveAsTable("retailfashiondata.rescued_data.invalid_cust_dim"))

## transformation: 
cust_dim = cust_dim.withColumn("age", col("age").cast("int"))\
    .withColumn("gender", when(col("gender") == "???", "NA").otherwise(col("gender"))) \
    .withColumn("email", when(col("email").isNull(), concat(lit("user"), col("customer_id").substr(2, 10).cast("int"), lit("@example.com"))).otherwise(col("email")))\
    .dropDuplicates(["customer_id"])\
    .drop("_rescued_data")

    

display(cust_dim)
cust_dim.printSchema()

##write:
(cust_dim.write
          .format("delta")
          .mode("append")
          .option("mergeSchema","true" )
        .saveAsTable("retailfashiondata.SilverTransformed.cust_dim"))


In [0]:
%sql
Select * from retailfashiondata.SilverTransformed.cust_dim where customer_id="C025010";

In [0]:
%sql
--delete from retailfashiondata.rescued_data.invalid_prod_dim;
--delete from retailfashiondata.SilverTransformed.prod_dim;

In [0]:
prod_dim= spark.table("retailfashiondata.silver.producttable")

## storing the _rescued_data to where it is not null(schema retailfashiondata.rescued_data)
invalid_prod_dim= prod_dim.filter(col("_rescued_data").isNotNull())

## write to invalid data
(invalid_prod_dim.write
                .format("delta")
                .mode("append")
                .option("mergeSchema","true")
                .saveAsTable("retailfashiondata.rescued_data.invalid_prod_dim"))


prod_dim= prod_dim.withColumn("category", when(col("category")=="???","NA").otherwise(col("category")))\
           .withColumn("color", when(col("color").isNull(),"NA").otherwise(col("color")))\
           .withColumn("supplier", when(col("supplier")=="suppliera","Supplier-A")\
                                  .when(col("supplier")=="supplierb","Supplier-B")\
                                  .when(col("supplier")=="supplierc","Supplier-C")\
                                  .when(col("supplier")=="supplierd","Supplier-D")\
                                 .otherwise(col("supplier"))
           )\
           .withColumn("cost_price",col("cost_price").cast("double"))\
           .withColumn("list_price",col("list_price").cast("double"))\
           .withColumn("profit_margin",round(col("List_price")-col("cost_price"),2))\
           .dropDuplicates(["product_id"])\
           .drop("_rescued_data")
           
                                      
                              
display(prod_dim)

##write:
(prod_dim.write
          .format("delta")
          .mode("overwrite")
          .option("mergeSchema","true")
        .saveAsTable("retailfashiondata.SilverTransformed.prod_dim"))

In [0]:
%sql
--delete from retailfashiondata.rescued_data.invalid_store_dim;
--delete from retailfashiondata.SilverTransformed.store_dim;

In [0]:
store_dim= spark.table("retailfashiondata.silver.storetable")

## storing the _rescued_data to where it is not null(schema retailfashiondata.rescued_data)
invalid_store_dim= store_dim.filter(col("_rescued_data").isNotNull())
                          
                        

(invalid_store_dim.write
                .format("delta")
                .mode("append")
                .option("mergeSchema","true")
                .saveAsTable("retailfashiondata.rescued_data.invalid_store_dim"))


store_dim= store_dim.withColumn("store_size_m2",col("store_size_m2").cast("int")) \
                    .drop("_rescued_data")

display(store_dim)

##write:
(store_dim.write
          .format("delta")
          .mode("append")
          .option("mergeSchema","true")
        .saveAsTable("retailfashiondata.SilverTransformed.store_dim"))

In [0]:
%sql
--delete from retailfashiondata.rescued_data.invalid_sales_fact;
--delete from retailfashiondata.SilverTransformed.sales_fact;

In [0]:
sales_fact = spark.table("retailfashiondata.silver.salestable")

## storing the _rescued_data to where it is not null(schema retailfashiondata.rescued_data)
invalid_sales_fact= sales_fact.filter(col("_rescued_data").isNotNull())

##write
(invalid_sales_fact.write
                .format("delta")
                .mode("append")
                .option("mergeSchema","true")
                .saveAsTable("retailfashiondata.rescued_data.invalid_sales_fact"))

###withColumn("discount",when(col("discount").isNull(),"NA").otherwise(col("discount")))\ -- converting dicount to double and it is not taking NA.

sales_fact= sales_fact.withColumn("customer_id",when(col("customer_id").isNull(),"NA").otherwise(col("customer_id")))\
                      .withColumn("date",col("date").cast("date"))\
                      .withColumn("quantity",col("quantity").cast("int"))\
                      .withColumn("discount",col("discount").cast("double"))\
                      .withColumn("returned",col("returned").cast("boolean"))\
                      .drop("_rescued_data")

display(sales_fact)

(sales_fact.write
            .format("delta")
            .mode("append")
            .option("mergeSchema","true")
            .saveAsTable("retailfashiondata.silvertransformed.sales_fact"))

In [0]:
%sql
select * from retailfashiondata.silvertransformed.sales_fact;

### DATA QUALITY CHECK

In [0]:
%sql
create schema retailfashiondata.DataCheckFails


In [0]:
%sql
--delete from retailfashiondata.DataCheckFails.dq_cust;
--delete from retailfashiondata.DataCheckFails.dq_prod;
--delete from retailfashiondata.DataCheckFails.dq_store_dim;
--delete from retailfashiondata.DataCheckFails.sq_sales_fact;


In [0]:
dq_cust= cust_dim\
    .withColumn("dq_customer_id",when(col("customer_id").isNull() | (col("customer_id")=="NA"),"INVALID").otherwise("VALID"))\
    .withColumn("dq_age",when(col("age").isNull(),"INVALID").otherwise("VALID"))\
    .withColumn("dq_gender",when(col("gender").isNull() | (col("gender")=="???"),"INVALID").otherwise("VALID"))\
    .withColumn("dq_email",when(col("email").isNull(),"INVALID").otherwise("VALID"))


display(dq_cust)

(
dq_cust.write
       .format("delta")
       .mode("append")
       .option("mergeSchema","true")
       .saveAsTable("retailfashiondata.DataCheckFails.dq_cust")
)
   

In [0]:
dq_prod = prod_dim\
    .withColumn("dq_product_id", when(col("product_id").isNull(),"INVALID").otherwise("VALID"))\
    .withColumn("dq_category", when(col("category").isNull() | (col("category")=="NA"),"INVALID").otherwise("VALID"))\
    .withColumn("dq_color", when(col("color").isNull() | (col("color")=="NA"),"INVALID").otherwise("VALID"))\
    .withColumn("dq_cost_price", when(col("cost_price").isNull(),"INVALID").otherwise("VALID"))\
    .withColumn("dq_list_price", when(col("list_price").isNull(),"INVALID").otherwise("VALID"))

display(dq_prod)

(dq_prod.write
   .format("delta")
   .mode("append")
   .option("mergeSchema","true")
   .saveAsTable("retailfashiondata.DataCheckFails.dq_prod"))

In [0]:
dq_store_dim= store_dim\
    .withColumn("dq_store_id", when((col("store_id").isNull()) | (col("store_id")=="NA"),"INVALID").otherwise("VALID"))\
    .withColumn("dq_store_name", when((col("store_name").isNull()) | (col("store_name")=="NA"),"INVALID").otherwise("VALID"))\
    .withColumn("dq_store_size_m2", when(col("store_size_m2").isNull(),"INVALID").otherwise("VALID"))
      
display(dq_store_dim)

(dq_store_dim.write
   .format("delta")
   .mode("append")
   .option("mergeSchema","true")
   .saveAsTable("retailfashiondata.DataCheckFails.dq_store_dim"))

In [0]:
dq_sales_fact = sales_fact\
                 .withColumn("dq_product_id", when((col("product_id").isNull()) | (col("product_id")=="NA"),"INVALID").otherwise("VALID"))\
                 .withColumn("dq_customer_id", when((col("customer_id").isNull()) | (col("customer_id")=="NA"),"INVALID").otherwise("VALID"))\
                 .withColumn("dq_store_id", when((col("store_id").isNull()) | (col("store_id")=="NA"),"INVALID").otherwise("VALID"))\
                 .withColumn("dq_discount", when(col("discount").isNull(),"INVALID").otherwise("VALID"))
                     
display(dq_sales_fact)

(dq_sales_fact.write
   .format("delta")
   .mode("append")
   .option("mergeSchema","true")
   .saveAsTable("retailfashiondata.DataCheckFails.sq_sales_fact"))

In [0]:
%sql
drop table retailfashiondata.DataCheckFails.dq_cust